# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata as an object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will enumerate all record sets found in the dataset (by their `@id`), and for each, list the associated fields (with their `@id`s and names) and columns.

In [ ]:
# List available record sets and their fields
record_sets = metadata.record_sets
print("Record sets (with '@id') found in dataset:\n")
for rs in record_sets:
    print(f"  RecordSet @id: {rs.id}")
    print(f"    Name: {rs.name}")
    print("    Fields (with '@id' and name):")
    for field in rs.fields:
        print(f"      - @id: {field.id} | name: {field.name}")
    print("    Columns (files with columns):")
    for file in rs.files:
        print(f"      FileObject @id: {file.id} | name: {file.name}")
        if hasattr(file, 'columns') and file.columns:
            for column in file.columns:
                print(f"        - Column @id: {column.id} | name: {column.name}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We will use the `@id` field to reference entities as shown above.

First, let's collect the list of record set `@id`s.

In [ ]:
# Gather all record set @id values
record_set_ids = [rs.id for rs in metadata.record_sets]
print("All record set @id list:\n", record_set_ids)

# Extract data from each record set into pandas DataFrames by @id
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded data for RecordSet @id: {record_set_id}, shape: {df.shape}")
    if not df.empty:
        print(f"Columns (@id for fields/columns):\n{df.columns.tolist()}")
        print(df.head(2))
        print("")

## 4. Exploratory Data Analysis (EDA)
Let's select a record set with substantial numeric data (e.g., protein features), filter on a numeric field using its `@id`, normalize it, and group by a categorical field (`@id`).

> **Important:** All references to data columns, fields, and grouping must use their `@id` (not display name or label), as shown above. Adjust field IDs below as appropriate for your dataset.

In [ ]:
# Example: Using the first Record Set
record_set_id = record_set_ids[0]  # Adjust to the relevant Record Set @id for your data analysis
df = dataframes[record_set_id]

# Inspect columns
print(f"RecordSet @id: {record_set_id} columns: {df.columns.tolist()}")

# Choose numeric and categorical fields by @id (displayed in previous Overviews)
# For demonstration, we try typical common field names; adjust as per your data
numeric_candidates = [col for col in df.columns if df[col].dtype in ['float64', 'int64'] or pd.api.types.is_numeric_dtype(df[col])]
print('Numeric field candidates:', numeric_candidates)
if not numeric_candidates:
    raise Exception("No numeric field found. Please adjust for your dataset.")
numeric_field_id = numeric_candidates[0]  # Use the first numeric field (@id)

threshold = df[numeric_field_id].mean()  # Use mean as threshold for demonstration
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / (filtered_df[numeric_field_id].std() + 1e-8)

print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to group by a likely categorical field
categorical_candidates = [col for col in df.columns if df[col].dtype == 'object' or pd.api.types.is_string_dtype(df[col])]
print('Categorical field candidates:', categorical_candidates)
group_field_id = None
if categorical_candidates:
    group_field_id = categorical_candidates[0]
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped (mean) data by {group_field_id} (for {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All visualizations should use column `@id` values.

Let's plot the distribution of the selected numeric field and a boxplot by the selected group field.

In [ ]:
# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if there is data
if not filtered_df.empty:
    plt.figure(figsize=(6, 4))
    sns.histplot(filtered_df[numeric_field_id], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id is not None and group_field_id in filtered_df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and process the protein mass spectrometry dataset using the Croissant schema and the `mlcroissant` library. 

**Key steps included:**
- Downloading and inspecting record sets, fields and their `@id`s.
- Loading record data into DataFrames for flexible processing.
- Filtering, normalizing, and grouping numeric data fields using only their `@id` for reliable reproducibility.
- Visualizing the distribution and relationships of key attributes.

You can further extend this notebook with domain-specific analyses using the same robust, schema-driven approach.